# Modelagem de tópicos probabilística com Latent Dirichlet Allocation (LDA)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK, Pandas  
- **Arquivo Original:** `exemploLDA.ipynb`  

---


# Topic Modeling com LDA
- Adaptado de https://github.com/kapadias/mediumposts/blob/master/natural_language_processing/topic_modeling/notebooks/Introduction%20to%20Topic%20Modeling.ipynb

In [ ]:
pip install --upgrade gensim

In [ ]:
!pip install pyLDAvis

In [ ]:
!pip install -U pandas-profiling

In [ ]:
pip install markupsafe==2.0.1

In [ ]:
import pandas as pd
import os
import re
from wordcloud import WordCloud
import gensim
from gensim.utils import simple_preprocess
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import gensim.corpora as corpora
from pprint import pprint
import pickle
from gensim.models import CoherenceModel

import pyLDAvis
import pyLDAvis.gensim_models

- Dataset 'papers.csv' contém aprox. 7240 texto de artigos da área de ML: https://www.kaggle.com/benhamner/nips-papers?select=papers.csv


In [8]:
papers = pd.read_csv('papers.csv')
papers.head()

,id,year,title,event_type,pdf_name,abstract,paper_text
0,1,1987,Self-Organization of Associative Database and ...,NaN,1-self-organization-of-associative-database-an...,Abstract Missing,767\n\nSELF-ORGANIZATION OF ASSOCIATIVE DATABA...
1,10,1987,A Mean Field Theory of Layer IV of Visual Cort...,NaN,10-a-mean-field-theory-of-layer-iv-of-visual-c...,Abstract Missing,683\n\nA MEAN FIELD THEORY OF LAYER IV OF VISU...
2,100,1988,Storing Covariance by the Associative Long-Ter...,NaN,100-storing-covariance-by-the-associative-long...,Abstract Missing,394\n\nSTORING COVARIANCE BY THE ASSOCIATIVE\n...
3,1000,1994,Bayesian Query Construction for Neural Network...,NaN,1000-bayesian-query-construction-for-neural-ne...,Abstract Missing,Bayesian Query Construction for Neural\nNetwor...
4,1001,1994,"Neural Network Ensembles, Cross Validation, an...",NaN,1001-neural-network-ensembles-cross-validation...,Abstract Missing,"Neural Network Ensembles, Cross\nValidation, a..."


In [9]:
# Remove colunas não utilizadas
papers = papers.drop(columns=['id', 'event_type', 'pdf_name'], axis=1).sample(100)
papers.head()

,year,title,abstract,paper_text
5267,2015,Biologically Inspired Dynamic Textures for Pro...,Perception is often described as a predictive ...,Biologically Inspired Dynamic Textures\nfor Pr...
1052,2001,Laplacian Eigenmaps and Spectral Techniques fo...,Abstract Missing,Laplacian Eigenmaps and Spectral\nTechniques f...
2045,2005,Integrate-and-Fire models with adaptation are ...,Abstract Missing,Integrate-and-Fire models with adaptation are\...
388,1997,Phase Transitions and the Perceptual Organizat...,Abstract Missing,Phase transitions and the perceptual\norganiza...
6078,2016,Crowdsourced Clustering: Querying Edges vs Tri...,We consider the task of clustering items using...,Crowdsourced Clustering: Querying Edges vs\nTr...


In [10]:
# Cria nova coluna para o texto preprocessado
# Remove pontuação
papers['paper_text_processed'] = papers['paper_text'].map(lambda x: re.sub('[,\.!?]', '', x))

# Converte para letras minusculas
papers['paper_text_processed'] = papers['paper_text_processed'].map(lambda x: x.lower())

papers['paper_text_processed'].head()

<input>:3: DeprecationWarning: invalid escape sequence \.
<input>:3: DeprecationWarning: invalid escape sequence \.
<input>:3: DeprecationWarning: invalid escape sequence \.
<input>:3: DeprecationWarning: invalid escape sequence \.
<input>:3: DeprecationWarning: invalid escape sequence \.
<input>:3: DeprecationWarning: invalid escape sequence \.
<ipython-input-10-a3f350cdc85c>:3: DeprecationWarning: invalid escape sequence \.
  papers['paper_text_processed'] = papers['paper_text'].map(lambda x: re.sub('[,\.!?]', '', x))


5267    biologically inspired dynamic textures\nfor pr...
1052    laplacian eigenmaps and spectral\ntechniques f...
2045    integrate-and-fire models with adaptation are\...
388     phase transitions and the perceptual\norganiza...
6078    crowdsourced clustering: querying edges vs\ntr...
Name: paper_text_processed, dtype: object

In [11]:
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))

def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]


data = papers.paper_text_processed.values.tolist()
data_words = list(sent_to_words(data))

# remove stopwords
data_words = remove_stopwords(data_words)

print(data_words[:1][0][:30])

['biologically', 'inspired', 'dynamic', 'textures', 'probing', 'motion', 'perception', 'andrew', 'isaac', 'meso', 'institut', 'de', 'neurosciences', 'de', 'la', 'timone', 'umr', 'cnrs', 'aix', 'marseille', 'universite', 'marseille', 'cedex', 'france', 'andrewmeso', 'univ', 'amufr', 'jonathan', 'vacher', 'cnrs']


In [12]:
# Cria dicionário
id2word = corpora.Dictionary(data_words)

# Corpus
texts = data_words

# Matriz termo-documento e gera o bag of words.
corpus = [id2word.doc2bow(text) for text in texts]

print(corpus[:1][0][:30])
# (id_token, freq)

[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 2), (6, 2), (7, 2), (8, 1), (9, 3), (10, 1), (11, 2), (12, 1), (13, 1), (14, 2), (15, 1), (16, 1), (17, 1), (18, 1), (19, 2), (20, 3), (21, 1), (22, 2), (23, 3), (24, 1), (25, 1), (26, 3), (27, 6), (28, 4), (29, 2)]


In [13]:
num_topics = 10

# Cria modelo LDA
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, passes=50)

# Print the Keyword in the 10 topics
pprint(lda_model.print_topics())
#doc_lda = lda_model[corpus]

[(0,
  '0.010*"distribution" + 0.008*"tree" + 0.007*"learning" + 0.006*"policy" + '
  '0.006*"label" + 0.006*"risk" + 0.006*"function" + 0.006*"sampling" + '
  '0.005*"hdp" + 0.005*"time"'),
 (1,
  '0.014*"data" + 0.014*"xt" + 0.007*"model" + 0.007*"time" + 0.005*"number" + '
  '0.005*"problem" + 0.005*"distribution" + 0.005*"set" + 0.005*"given" + '
  '0.004*"inference"'),
 (2,
  '0.008*"learning" + 0.007*"data" + 0.007*"class" + 0.007*"examples" + '
  '0.006*"algorithm" + 0.006*"phase" + 0.006*"model" + 0.005*"image" + '
  '0.005*"set" + 0.005*"classes"'),
 (3,
  '0.009*"algorithm" + 0.007*"model" + 0.005*"data" + 0.005*"means" + '
  '0.005*"time" + 0.004*"algorithms" + 0.004*"function" + 0.004*"learning" + '
  '0.004*"one" + 0.004*"log"'),
 (4,
  '0.010*"model" + 0.006*"figure" + 0.005*"models" + 0.005*"log" + '
  '0.005*"triangle" + 0.005*"edge" + 0.005*"spatial" + 0.005*"images" + '
  '0.004*"one" + 0.004*"algorithm"'),
 (5,
  '0.009*"learning" + 0.009*"training" + 0.007*"features

In [14]:
# Visualização dos tópicos
pyLDAvis.enable_notebook()

LDAvis_data_filepath = os.path.join(''+str(num_topics))

# # this is a bit time consuming - make the if statement True
# # if you want to execute visualization prep yourself
if 1 == 1:
    LDAvis_prepared = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word)
    with open(LDAvis_data_filepath, 'wb') as f:
        pickle.dump(LDAvis_prepared, f)

# load the pre-prepared pyLDAvis data from disk
with open(LDAvis_data_filepath, 'rb') as f:
    LDAvis_prepared = pickle.load(f)

pyLDAvis.save_html(LDAvis_prepared, ''+ str(num_topics) +'.html')

LDAvis_prepared

/usr/local/lib/python3.7/dist-packages/pyLDAvis/_prepare.py:247: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  by='saliency', ascending=False).head(R).drop('saliency', 1)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.021653 -0.035758       1        1  19.138601
6      0.064794  0.049511       2        1  14.371299
5      0.025453  0.148361       3        1  12.436718
9      0.010482  0.006343       4        1  12.327747
8      0.023042 -0.032537       5        1  10.928765
4     -0.028251 -0.053228       6        1   8.384853
1      0.048127 -0.057576       7        1   6.976740
0      0.064773 -0.070584       8        1   6.377133
2      0.014002  0.044211       9        1   5.615723
7     -0.244077  0.001257      10        1   3.442422, topic_info=              Term         Freq        Total Category  logprob  loglift
1371        kernel   470.000000   470.000000  Default  30.0000  30.0000
3223            xt   392.000000   392.000000  Default  29.0000  29.0000
1101            vi   209.000000   209.000000  Default  28.0000  28.0000
635          model  1406.000000  1406.000000  Default  27.0000  27.0000
285   distribution   582.000000   582.000000  Default  26.0000  26.0000
...            ...          ...          ...      ...      ...      ...
704          order    55.903444   365.460849  Topic10  -4.9727   1.4915
896         second    43.067032   324.195776  Topic10  -5.2335   1.3504
388          first    45.978033   587.581640  Topic10  -5.1681   0.8211
748          phase    31.042353   166.410720  Topic10  -5.5609   1.6899
933         simple    30.388040   224.215308  Topic10  -5.5823   1.3704

[763 rows x 6 columns], token_table=       Topic      Freq     Term
term                           
13195      9  0.929691  abalone
4471       8  0.859949  acrobot
2911       1  0.013346   active
2911       2  0.126790   active
2911       3  0.106771   active
...      ...       ...      ...
3797       1  0.156846       zt
3797       5  0.147620       zt
3797       6  0.166073       zt
3797       7  0.055358       zt
3797       8  0.488991       zt

[2336 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[4, 7, 6, 10, 9, 5, 2, 1, 3, 8])

In [15]:
# Computar o Score de Coerência
coherence_model_lda = CoherenceModel(model=lda_model, texts=texts, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('Score de Coerência: ', coherence_lda)

Score de Coerência:  0.2941399558034361


In [16]:
lda_model.print_topics(num_topics = 10, num_words = 5)

[(0,
  '0.010*"distribution" + 0.008*"tree" + 0.007*"learning" + 0.006*"policy" + 0.006*"label"'),
 (1,
  '0.014*"data" + 0.014*"xt" + 0.007*"model" + 0.007*"time" + 0.005*"number"'),
 (2,
  '0.008*"learning" + 0.007*"data" + 0.007*"class" + 0.007*"examples" + 0.006*"algorithm"'),
 (3,
  '0.009*"algorithm" + 0.007*"model" + 0.005*"data" + 0.005*"means" + 0.005*"time"'),
 (4,
  '0.010*"model" + 0.006*"figure" + 0.005*"models" + 0.005*"log" + 0.005*"triangle"'),
 (5,
  '0.009*"learning" + 0.009*"training" + 0.007*"features" + 0.007*"data" + 0.006*"network"'),
 (6,
  '0.012*"kernel" + 0.009*"data" + 0.009*"learning" + 0.007*"function" + 0.006*"set"'),
 (7,
  '0.014*"vi" + 0.010*"cells" + 0.009*"motion" + 0.009*"information" + 0.009*"game"'),
 (8,
  '0.015*"model" + 0.008*"data" + 0.006*"network" + 0.006*"models" + 0.006*"distribution"'),
 (9,
  '0.007*"time" + 0.006*"model" + 0.006*"spike" + 0.005*"algorithm" + 0.005*"function"')]

In [17]:
lda_model.print_topic(0)

'0.010*"distribution" + 0.008*"tree" + 0.007*"learning" + 0.006*"policy" + 0.006*"label" + 0.006*"risk" + 0.006*"function" + 0.006*"sampling" + 0.005*"hdp" + 0.005*"time"'